# ✈️ Flight Fare Prediction — AI & Machine Learning System
## Master Notebook: Full Dataset — 10-Fold Cross-Validation (**Comprehensive Submission**)

---
**Course:** AIC354 — Machine Learning Fundamentals  
**Assignment:** 3 — Applying, Evaluating & Comparing Multiple ML Algorithms on a Real-World Dataset  
**Dataset:** Flight Fare Prediction (10,683 airline travel records)  

---
### 🤖 5 Machine Learning Algorithms Compared

| # | Algorithm | Category | Key Hyperparameters |
|---|-----------|----------|---------------------|
| 1 | **Ridge Regression** | Regularized Linear (L2) | `alpha=1.0`, StandardScaler |
| 2 | **Lasso Regression** | Regularized Linear (L1) | `alpha=0.1`, StandardScaler |
| 3 | **Support Vector Regression (SVR)** | Kernel-Based | `kernel=rbf`, `C=100`, StandardScaler |
| 4 | **Extra Trees Regressor** | Ensemble (Bagging) | `n_estimators=200`, `max_features='sqrt'` |
| 5 | **XGBoost Regressor** | Ensemble (Boosting) | `n_estimators=200`, `lr=0.05`, `max_depth=6` |

---
> **Evaluation Metric:** RMSE (Root Mean Squared Error) — lower is better  
> **Cross-Validation:** 10-Fold on **full dataset** (10,683 instances)  
> **Deployment Target:** Streamlit Community Cloud (Production Ready)  


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 1: Install Required Libraries
# ══════════════════════════════════════════════════════════════
!pip install xgboost tqdm openpyxl --quiet
print('✅ Dependencies successfully installed.')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 2: Import Libraries & Styling Setup
# ══════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import time
import pickle
import os

from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import ExtraTreesRegressor
from xgboost import XGBRegressor

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

from tqdm.notebook import tqdm
from IPython.display import display, HTML

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.family'] = 'sans-serif'

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('✅ All libraries imported successfully!')
print(f'   NumPy     : {np.__version__}')
print(f'   Pandas    : {pd.__version__}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 3: Load Dataset (CSV or Excel)
# ══════════════════════════════════════════════════════════════
DATA_PATHS = [
    'Data_Train.csv',
    'Data_Train.xlsx',
    '../dataset/Data_Train.csv',
    '../dataset/Data_Train.xlsx',
    'dataset/Data_Train.csv',
    'dataset/Data_Train.xlsx'
]

df = None
for p in DATA_PATHS:
    if os.path.exists(p):
        if p.endswith('.csv'):
            df = pd.read_csv(p)
        else:
            df = pd.read_excel(p)
        print(f'📂 Loaded dataset from local path: {p}')
        break

if df is None and IN_COLAB:
    print('📂 Please upload Data_Train.csv or Data_Train.xlsx:')
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    df = pd.read_csv(filename) if filename.endswith('.csv') else pd.read_excel(filename)

print(f'\n✅ Flight dataset loaded successfully!')
print(f'   Records  : {df.shape[0]:,} rows')
print(f'   Features : {df.shape[1]} columns')
print(f'   Columns  : {list(df.columns)}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 4: Exploratory Data Analysis (EDA) — Overview
# ══════════════════════════════════════════════════════════════
print('=' * 75)
print('  SECTION 1: EXPLORATORY DATA ANALYSIS (EDA)')
print('=' * 75)

print('\n📋 First 5 Rows:')
display(df.head())

print('\n📊 Dataset Info:')
print(df.info())

print('\n🔍 Missing Values Before Cleaning:')
print(df.isnull().sum())

print('\n📈 Target Price Summary (INR ₹):')
display(df['Price'].describe().to_frame())

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 5: Exploratory Visualizations
# ══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('Exploratory Data Analysis — Flight Fare Dataset', fontsize=16, fontweight='bold')

# 1. Price Distribution
axes[0, 0].hist(df['Price'], bins=50, color='#0284c7', edgecolor='white', alpha=0.85)
axes[0, 0].axvline(df['Price'].mean(), color='#ef4444', linestyle='--', linewidth=2, label=f'Mean = ₹{df["Price"].mean():,.0f}')
axes[0, 0].axvline(df['Price'].median(), color='#10b981', linestyle=':', linewidth=2, label=f'Median = ₹{df["Price"].median():,.0f}')
axes[0, 0].set_title('Flight Price Distribution (INR ₹)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Price (₹)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].legend()

# 2. Mean Price by Airline
airline_means = df.groupby('Airline')['Price'].mean().sort_values(ascending=False)
axes[0, 1].barh(airline_means.index[:8], airline_means.values[:8], color='#6366f1', edgecolor='white')
axes[0, 1].set_title('Top 8 Airlines by Mean Ticket Price', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Average Price (₹)')
axes[0, 1].invert_yaxis()

# 3. Total Stops vs Price
stops_means = df.groupby('Total_Stops')['Price'].mean().sort_values()
axes[1, 0].bar(stops_means.index.astype(str), stops_means.values, color='#14b8a6', edgecolor='white')
axes[1, 0].set_title('Average Price by Total Stops', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Number of Stops')
axes[1, 0].set_ylabel('Mean Price (₹)')
axes[1, 0].tick_params(axis='x', rotation=15)

# 4. Source Airport Flight Count
source_counts = df['Source'].value_counts()
axes[1, 1].pie(source_counts, labels=source_counts.index, autopct='%1.1f%%',
              colors=['#38bdf8', '#818cf8', '#34d399', '#f472b6', '#fbbf24'],
              wedgeprops={'edgecolor': 'white'})
axes[1, 1].set_title('Flight Traffic Distribution by Source City', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('eda_flight_plots.png', bbox_inches='tight', dpi=150)
plt.show()
print('\n📊 Exploratory charts saved as eda_flight_plots.png')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 6: Feature Engineering & Preprocessing Pipeline
# ══════════════════════════════════════════════════════════════
print('=' * 75)
print('  SECTION 2: FEATURE ENGINEERING & DATA PREPARATION')
print('=' * 75)

df_proc = df.copy()

# 1. Clean missing values
df_proc.dropna(subset=['Total_Stops', 'Route'], inplace=True)

# 2. Standardize Destination names ('New Delhi' -> 'Delhi')
df_proc['Destination'] = df_proc['Destination'].replace('New Delhi', 'Delhi')

# 3. Parse Date_of_Journey
df_proc['Journey_Day'] = df_proc['Date_of_Journey'].apply(lambda x: int(str(x).split('/')[0]))
df_proc['Journey_Month'] = df_proc['Date_of_Journey'].apply(lambda x: int(str(x).split('/')[1]))
dt_series = pd.to_datetime(df_proc['Date_of_Journey'], format='%d/%m/%Y', errors='coerce')
df_proc['Journey_DayOfWeek'] = dt_series.dt.dayofweek.fillna(0).astype(int)

# 4. Parse Dep_Time
df_proc['Dep_Hour'] = df_proc['Dep_Time'].apply(lambda x: int(str(x).split(':')[0]))
df_proc['Dep_Min'] = df_proc['Dep_Time'].apply(lambda x: int(str(x).split(':')[1]))

# 5. Parse Arrival_Time
def parse_arr_time(val):
    clean = str(val).split()[0]
    h, m = clean.split(':')
    return int(h), int(m)

arr_parsed = df_proc['Arrival_Time'].apply(parse_arr_time)
df_proc['Arrival_Hour'] = [x[0] for x in arr_parsed]
df_proc['Arrival_Min'] = [x[1] for x in arr_parsed]

# 6. Parse Duration to Minutes
def parse_duration(d_str):
    hours, mins = 0, 0
    for token in str(d_str).strip().split():
        if 'h' in token:
            hours = int(token.replace('h', ''))
        elif 'm' in token:
            mins = int(token.replace('m', ''))
    return hours * 60 + mins

df_proc['Duration_Mins'] = df_proc['Duration'].apply(parse_duration)

# 7. Map Total_Stops to numeric integers
stops_map = {
    'non-stop': 0,
    '1 stop': 1,
    '2 stops': 2,
    '3 stops': 3,
    '4 stops': 4
}
df_proc['Total_Stops'] = df_proc['Total_Stops'].map(stops_map).fillna(0).astype(int)

# 8. Categorical Encoding with LabelEncoders
CAT_COLS = ['Airline', 'Source', 'Destination']
le_dict = {}
for col in CAT_COLS:
    le = LabelEncoder()
    df_proc[col] = le.fit_transform(df_proc[col].astype(str))
    le_dict[col] = le

# Drop redundant / raw text columns
DROP_COLS = ['Date_of_Journey', 'Route', 'Dep_Time', 'Arrival_Time', 'Duration', 'Additional_Info', 'Price']
FEATURE_COLS = [c for c in df_proc.columns if c not in DROP_COLS]

X = df_proc[FEATURE_COLS].copy()
y = df_proc['Price'].copy()

print(f'✅ Feature Engineering Complete!')
print(f'   Final Samples : {X.shape[0]:,}')
print(f'   Features ({len(FEATURE_COLS)})  : {FEATURE_COLS}')
print(f'   Target Range  : ₹{y.min():,} to ₹{y.max():,} (Mean = ₹{y.mean():,.2f})')
display(X.head())

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 7: Correlation Heatmap
# ══════════════════════════════════════════════════════════════
plt.figure(figsize=(11, 8))
corr_df = pd.concat([X, y], axis=1).corr()
mask = np.triu(np.ones_like(corr_df, dtype=bool))
sns.heatmap(corr_df, mask=mask, annot=True, fmt='.2f', cmap='Blues', linewidths=0.5)
plt.title('Correlation Matrix of Engineered Flight Features & Price', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_flight_matrix.png', bbox_inches='tight', dpi=150)
plt.show()
print('\n🔥 Correlation matrix saved.')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 8: Define the 5 ML Algorithms (with Pipelines)
# ══════════════════════════════════════════════════════════════
print('=' * 75)
print('  SECTION 3: ALGORITHM SPECIFICATION (10-FOLD CV READY)')
print('=' * 75)

models = {
    '① Ridge Regression'      : Pipeline([
        ('scaler', StandardScaler()),
        ('model',  Ridge(alpha=1.0))
    ]),
    '② Lasso Regression'      : Pipeline([
        ('scaler', StandardScaler()),
        ('model',  Lasso(alpha=0.1, max_iter=10000))
    ]),
    '③ SVR (RBF Kernel)'      : Pipeline([
        ('scaler', StandardScaler()),
        ('model',  SVR(kernel='rbf', C=100, gamma='scale', epsilon=0.1))
    ]),
    '④ Extra Trees Regressor' : ExtraTreesRegressor(
        n_estimators=200, max_features='sqrt',
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    '⑤ XGBoost Regressor'     : XGBRegressor(
        n_estimators=200, learning_rate=0.05,
        max_depth=6, subsample=0.8,
        colsample_bytree=0.8, reg_alpha=0.1,
        random_state=RANDOM_STATE, verbosity=0,
        eval_metric='rmse', n_jobs=-1
    ),
}

for name in models:
    print(f'   ✔ Configured: {name}')
print(f'\n✅ {len(models)} Distinct algorithms prepared for 10-Fold Cross-Validation!')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 9: 10-Fold Cross-Validation Execution
# ══════════════════════════════════════════════════════════════
print('=' * 75)
print('  SECTION 4: 10-FOLD CROSS-VALIDATION ON FULL DATASET')
print('=' * 75)

K_FOLDS = 10
kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=RANDOM_STATE)

results = {}
total_start = time.time()

print(f'\n🔁 Executing {K_FOLDS}-Fold CV on {X.shape[0]:,} Flight Records...\n')

for name, model in tqdm(models.items(), desc='🚀 Training & Validating', unit='model'):
    start = time.time()
    
    # Negative MSE -> RMSE
    neg_mse = cross_val_score(model, X, y, cv=kf,
                              scoring='neg_mean_squared_error', n_jobs=-1)
    rmse_scores = np.sqrt(-neg_mse)
    
    # R² Score
    r2_scores = cross_val_score(model, X, y, cv=kf,
                                scoring='r2', n_jobs=-1)
    
    # Mean Absolute Error
    neg_mae = cross_val_score(model, X, y, cv=kf,
                              scoring='neg_mean_absolute_error', n_jobs=-1)
    mae_scores = -neg_mae
    
    elapsed = time.time() - start
    
    results[name] = {
        'RMSE Mean': round(float(rmse_scores.mean()), 2),
        'RMSE Std' : round(float(rmse_scores.std()), 2),
        'R² Mean'  : round(float(r2_scores.mean()), 4),
        'R² Std'   : round(float(r2_scores.std()), 4),
        'MAE Mean' : round(float(mae_scores.mean()), 2),
        'Time (s)' : round(float(elapsed), 2),
        'All RMSE' : rmse_scores,
    }
    
    clean_name = name.split(' ', 1)[1]
    print(f'  ✔ {clean_name:<26} RMSE: ₹{rmse_scores.mean():,.2f} ± {rmse_scores.std():,.2f}  |  R²: {r2_scores.mean():.4f}  ({elapsed:.1f}s)')

total_elapsed = time.time() - total_start
print(f'\n⏱ Total CV Execution Time: {total_elapsed:.1f}s')
print('✅ 10-Fold Cross-Validation successfully completed!')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 10: Model Comparison Leaderboard Table
# ══════════════════════════════════════════════════════════════
print('=' * 75)
print('  SECTION 5: MODEL COMPARISON LEADERBOARD (10-FOLD CV)')
print('=' * 75)

summary = {}
for name, r in results.items():
    clean = name.split(' ', 1)[1]
    summary[clean] = {
        'RMSE Mean (₹)': r['RMSE Mean'],
        'RMSE Std (₹)' : r['RMSE Std'],
        'R² Score'     : r['R² Mean'],
        'MAE Mean (₹)' : r['MAE Mean'],
        'Time (s)'     : r['Time (s)'],
    }

results_df = pd.DataFrame(summary).T.reset_index()
results_df.columns = ['Model', 'RMSE Mean (₹)', 'RMSE Std (₹)', 'R² Score', 'MAE Mean (₹)', 'Time (s)']
results_df = results_df.sort_values('RMSE Mean (₹)').reset_index(drop=True)
results_df.insert(0, 'Rank', range(1, len(results_df) + 1))

def highlight_top(s):
    return ['background-color: #d1fae5; font-weight: bold' if i == 0 else '' for i in range(len(s))]

print('\n📋 10-Fold Cross-Validation Results Summary:')
display(results_df.style.apply(highlight_top).format({
    'RMSE Mean (₹)': '₹{:,.2f}',
    'RMSE Std (₹)' : '₹{:,.2f}',
    'R² Score'     : '{:.4f}',
    'MAE Mean (₹)' : '₹{:,.2f}',
    'Time (s)'     : '{:.2f}s'
}))

best = results_df.iloc[0]
print(f'\n🏆 Top-Performing Estimator: {best["Model"]}')
print(f'   RMSE: ₹{best["RMSE Mean (₹)"]:,.2f}  |  R²: {best["R² Score"]:.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 11: Comparison Visualizations
# ══════════════════════════════════════════════════════════════
PALETTE = ['#10b981', '#38bdf8', '#818cf8', '#f59e0b', '#ef4444']
colors = [PALETTE[0]] + PALETTE[1:len(results_df)]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Flight Fare Prediction — 10-Fold CV Performance Visualizer\n(Full Dataset: 10,682 Records)',
             fontsize=15, fontweight='bold', y=1.01)

model_names = results_df['Model'].tolist()

# 1. RMSE Mean Bar Chart
axes[0, 0].barh(model_names, results_df['RMSE Mean (₹)'], xerr=results_df['RMSE Std (₹)'],
                color=colors, edgecolor='white', capsize=4)
axes[0, 0].set_xlabel('RMSE Mean (₹) — Lower is Better', fontsize=11)
axes[0, 0].set_title('10-Fold CV RMSE Mean with Std Dev', fontsize=12, fontweight='bold')
axes[0, 0].invert_yaxis()
for i, (val, std) in enumerate(zip(results_df['RMSE Mean (₹)'], results_df['RMSE Std (₹)'])):
    axes[0, 0].text(val + std + 100, i, f'₹{val:,.0f}', va='center', fontsize=9)

# 2. R² Score
axes[0, 1].barh(model_names, results_df['R² Score'], color=colors, edgecolor='white')
axes[0, 1].set_xlabel('R² Score — Higher is Better', fontsize=11)
axes[0, 1].set_title('Coefficient of Determination (R²)', fontsize=12, fontweight='bold')
axes[0, 1].invert_yaxis()
for i, val in enumerate(results_df['R² Score']):
    axes[0, 1].text(val + 0.01, i, f'{val:.4f}', va='center', fontsize=9)

# 3. Boxplot across 10 folds
fold_data = [results[k]['All RMSE'] for k in models]
clean_labs = [k.split(' ', 1)[1] for k in models]
bp = axes[1, 0].boxplot(fold_data, patch_artist=True, vert=True, medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
axes[1, 0].set_xticklabels([l.replace(' ', '\n') for l in clean_labs], fontsize=8)
axes[1, 0].set_ylabel('Fold RMSE (₹)', fontsize=11)
axes[1, 0].set_title('RMSE Distribution across 10 Folds', fontsize=12, fontweight='bold')

# 4. Training Latency
axes[1, 1].bar(model_names, results_df['Time (s)'], color='#334155', edgecolor='white')
axes[1, 1].set_ylabel('Training Time (seconds)', fontsize=11)
axes[1, 1].set_title('Computational Efficiency (Training Latency)', fontsize=12, fontweight='bold')
axes[1, 1].tick_params(axis='x', rotation=20)
for i, val in enumerate(results_df['Time (s)']):
    axes[1, 1].text(i, val + 0.5, f'{val:.1f}s', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('comparison_flight_models.png', bbox_inches='tight', dpi=150)
plt.show()
print('\n📊 Comparison visualizer saved as comparison_flight_models.png')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 12: Train Best Model on Full Dataset & Export Bundle
# ══════════════════════════════════════════════════════════════
print('=' * 75)
print('  SECTION 6: TRAIN BEST MODEL ON FULL DATASET & SAVE')
print('=' * 75)

best_model_name = best['Model']
print(f'\n🏆 Selected Top Model: {best_model_name}')

# Retrieve model object
best_model_obj = None
for name, m in models.items():
    if name.split(' ', 1)[1] == best_model_name:
        best_model_obj = m
        break

print('\n🔄 Fitting best model on all 10,682 instances...')
best_model_obj.fit(X, y)

MODEL_DIR = 'models'
os.makedirs(MODEL_DIR, exist_ok=True)
MODEL_PATH = os.path.join(MODEL_DIR, 'best_model.pkl')

save_bundle = {
    'model'     : best_model_obj,
    'le_dict'   : le_dict,
    'features'  : list(X.columns),
    'target'    : 'Price',
    'model_name': best_model_name,
    'rmse'      : float(best['RMSE Mean (₹)']),
    'r2'        : float(best['R² Score']),
}

with open(MODEL_PATH, 'wb') as f:
    pickle.dump(save_bundle, f)

print(f'\n✅ Best Model successfully pickled to: {MODEL_PATH}')
print(f'   Bundle Size : {os.path.getsize(MODEL_PATH) / (1024 * 1024):.2f} MB')
print(f'   Included Components : model, le_dict, features, target, model_name')

if IN_COLAB:
    files.download(MODEL_PATH)
    print('📥 Downloaded best_model.pkl to your machine!')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 13: Feature Importance Analysis
# ══════════════════════════════════════════════════════════════
print('=' * 75)
print('  SECTION 7: FEATURE IMPORTANCE ANALYSIS')
print('=' * 75)

def extract_importance(model, feature_names):
    m = model
    if hasattr(model, 'named_steps'):
        m = model.named_steps.get('model', model)
    if hasattr(m, 'feature_importances_'):
        return m.feature_importances_
    elif hasattr(m, 'coef_'):
        return np.abs(m.coef_)
    return None

et_model = models['④ Extra Trees Regressor']
xgb_model = models['⑤ XGBoost Regressor']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Feature Importance — Tree-Based Ensembles', fontsize=14, fontweight='bold')

for ax, mod, title in [(axes[0], et_model, 'Extra Trees Regressor'),
                       (axes[1], xgb_model, 'XGBoost Regressor')]:
    imp = extract_importance(mod, FEATURE_COLS)
    if imp is not None:
        fi_df = pd.DataFrame({'Feature': FEATURE_COLS, 'Importance': imp})
        fi_df = fi_df.sort_values('Importance', ascending=True)
        ax.barh(fi_df['Feature'], fi_df['Importance'], color='#0284c7', edgecolor='white')
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xlabel('Relative Importance Score')
    else:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center')

plt.tight_layout()
plt.savefig('feature_importance_flight.png', bbox_inches='tight', dpi=150)
plt.show()
print('\n📊 Feature importance plots saved.')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 14: Verification Test & Sample Prediction
# ══════════════════════════════════════════════════════════════
print('=' * 75)
print('  SECTION 8: VERIFICATION TEST & SAMPLE PREDICTION')
print('=' * 75)

with open(MODEL_PATH, 'rb') as f:
    loaded_bundle = pickle.load(f)

loaded_model = loaded_bundle['model']
loaded_le = loaded_bundle['le_dict']
loaded_feats = loaded_bundle['features']

# Sample flight: IndiGo flight from Banglore to Delhi, 0 stops, 170 mins
sample_airline = 'IndiGo'
sample_source  = 'Banglore'
sample_dest    = 'Delhi'

sample_input = pd.DataFrame([{
    'Airline': loaded_le['Airline'].transform([sample_airline])[0],
    'Source': loaded_le['Source'].transform([sample_source])[0],
    'Destination': loaded_le['Destination'].transform([sample_dest])[0],
    'Total_Stops': 0,
    'Journey_Day': 24,
    'Journey_Month': 3,
    'Journey_DayOfWeek': 6,
    'Dep_Hour': 22,
    'Dep_Min': 20,
    'Arrival_Hour': 1,
    'Arrival_Min': 10,
    'Duration_Mins': 170
}])[loaded_feats]

pred_fare = loaded_model.predict(sample_input)[0]
print(f'✈️ Sample Flight: {sample_airline} | {sample_source} ➔ {sample_dest} (Non-stop, 2h 50m)')
print(f'💰 Predicted Fare : ₹{pred_fare:,.2f}')
print('✅ Verification test passed! Model is ready for Streamlit Web App.')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 15: Final Summary & Deployment Ready Checklist
# ══════════════════════════════════════════════════════════════
print('\n' + '═' * 75)
print('  FINAL SUMMARY REPORT — FLIGHT FARE PREDICTION AI SYSTEM')
print('═' * 75)
print(f'  Dataset     : Flight Fare Prediction (India Domestic Routes)')
print(f'  Records     : {X.shape[0]:,} verified flights')
print(f'  Features    : {len(FEATURE_COLS)} engineered predictive attributes')
print(f'  Target      : Flight Ticket Fare (INR ₹)')
print(f'  Validation  : 10-Fold Cross-Validation (Zero Data Leakage)')
print('─' * 75)
print('  ALGORITHM LEADERBOARD:')
print('─' * 75)
for _, row in results_df.iterrows():
    badge = '🥇' if row['Rank'] == 1 else ('🥈' if row['Rank'] == 2 else ('🥉' if row['Rank'] == 3 else f'  {int(row["Rank"])}.'))
    print(f'  {badge} {row["Model"]:<26} RMSE: ₹{row["RMSE Mean (₹)"]:,.2f}  |  R²: {row["R² Score"]:.4f}')
print('─' * 75)
print(f'  🏆 WINNING MODEL : {best["Model"]}')
print(f'     Validation RMSE : ₹{best["RMSE Mean (₹)"]:,.2f}')
print(f'     R² Score        : {best["R² Score"]:.4f}')
print('═' * 75)
print('\n🚀 STREAMLIT DEPLOYMENT INSTRUCTIONS:')
print('  1. Place the generated best_model.pkl in the models/ directory.')
print('  2. Push the repository to GitHub.')
print('  3. Log in to https://share.streamlit.io and select your repository.')
print('  4. Set Main file path: app.py and click Deploy! 🎉')
print('═' * 75)